# Capítulo 16 · Laboratório 16: força relativa
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance e do Banco Central na sua máquina e guarda uma cópia local (`dados_cap16.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O laboratório indexa o Ibovespa ao dólar e ao IPCA; monta razões entre ações (CSNA3 / GGBR4 e ITUB4 / BBDC4); calcula a força relativa de Mansfield de 66 ações do Ibovespa; testa o momentum 12-1 em carteiras mensais; monta uma grade de nove didática (a ideia de Ralph Acampora, com regras públicas, e não o ARGON) e mede o fôlego da força relativa.

In [ ]:
!pip install -q yfinance

In [ ]:
import json
import logging
import os
import urllib.request
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap16.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=1: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ACOES = ["ABEV3", "ALOS3", "ASAI3", "AZZA3", "B3SA3", "BBAS3", "BBDC4", "BBSE3", "BEEF3", "BPAC11", "BRAV3",
         "MBRF3", "CMIG4", "CMIN3", "CPFE3", "CSAN3", "CSNA3", "CXSE3", "CYRE3", "DIRR3", "EGIE3",
         "AXIA3", "EMBJ3", "ENEV3", "ENGI11", "EQTL3", "FLRY3", "GGBR4", "HAPV3", "HYPE3", "IGTI11", "IRBR3",
         "ISAE4", "ITUB4", "KLBN11", "LREN3", "MGLU3", "MOTV3", "MRVE3", "MULT3", "NATU3", "PCAR3", "PETR4",
         "PRIO3", "PSSA3", "RADL3", "RAIL3", "RAIZ4", "RDOR3", "RENT3", "SANB11", "SBSP3", "SLCE3", "SMFT3",
         "SUZB3", "TAEE11", "TIMS3", "TOTS3", "UGPA3", "USIM5", "VALE3", "VBBR3", "VIVA3", "VIVT3", "WEGE3", "YDUQ3"]
TICK = [f"{t}.SA" for t in ACOES]


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):
    D = pd.read_pickle(CACHE)
else:
    d = yf.download(TICK + ["^BVSP", "BRL=X"], start="2005-01-01", end=FIM, auto_adjust=True, progress=False,
                    group_by="column")
    confere({"as ações, o Ibovespa e o dólar": d})
    url = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"
           "&dataInicial=01/01/2005&dataFinal=30/09/2026")
    j = json.load(urllib.request.urlopen(url, timeout=30))
    ipca = pd.Series([float(x["valor"]) for x in j], index=pd.to_datetime([x["data"] for x in j], dayfirst=True))
    D = {"Close": d["Close"], "High": d["High"], "Low": d["Low"], "Open": d["Open"], "Volume": d["Volume"],
         "IPCA": ipca}
    pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
FECH = D["Close"][TICK].where(D["Volume"][TICK] > 0)          # pregão sem volume vira lacuna
FECH["^BVSP"] = D["Close"]["^BVSP"]                            # índice e câmbio não têm volume no Yahoo
FECH["BRL=X"] = D["Close"]["BRL=X"]
FECH = FECH.dropna(how="all")
MENSAL = FECH.resample("ME").last()
SEMANAL = FECH.resample("W-FRI").last()
print(f"Parte 1. {FECH[TICK].notna().any().sum()} ações com dados, Ibovespa e dólar desde {FECH.index[0]:%Y};"
      f" último pregão {FECH.index[-1]:%d/%m/%Y}; IPCA até {D['IPCA'].index[-1]:%m/%Y}.")

## Parte 2: indexação

In [ ]:
INDICE = (1 + D["IPCA"] / 100).cumprod()
INDICE.index = INDICE.index + pd.offsets.MonthEnd(0)
BASE = pd.DataFrame({"reais": MENSAL["^BVSP"], "dolares": MENSAL["^BVSP"] / MENSAL["BRL=X"],
                     "real_ipca": MENSAL["^BVSP"] / INDICE}).dropna()
TAB_IDX = []
print(f"\nParte 2. Ibovespa: variação desde cada data até {BASE.index[-1]:%m/%Y} (último IPCA divulgado)")
for ini in ["2008-05-31", "2011-01-31", "2021-06-30"]:
    a, b = BASE.loc[:ini].iloc[-1], BASE.iloc[-1]
    v = [b[c] / a[c] - 1 for c in BASE.columns]
    TAB_IDX.append((BASE.loc[:ini].index[-1], *v))
    print(f"   desde {BASE.loc[:ini].index[-1]:%m/%Y}: em reais {ps(v[0], 0)} | em dólares {ps(v[1], 0)} |"
          f" descontado o IPCA {ps(v[2], 0)}")

In [ ]:
import matplotlib.pyplot as plt
b = BASE / BASE.loc["2008-05-31"] * 100
b.plot(figsize=(8, 3.5), color=["black", "gray", "silver"])
plt.axhline(100, color="black", lw=0.5)
plt.title("Ibovespa em reais, em dólares e descontado o IPCA (maio de 2008 = 100)")
plt.show()

## Parte 3: pangaré ou alazão

In [ ]:
def razao(a, b):
    r = (FECH[f"{a}.SA"] / FECH[f"{b}.SA"]).dropna()
    return r


R_CG = razao("CSNA3", "GGBR4")
print(f"\nParte 3. Razão CSNA3 / GGBR4: {nr(R_CG.iloc[-1], 3)} em {R_CG.index[-1]:%d/%m/%Y}; em 12 meses"
      f" {ps(R_CG.iloc[-1] / R_CG.iloc[-253] - 1, 0)} (subindo = CSNA3 melhor que GGBR4)")
R_IB = razao("ITUB4", "BBDC4").resample("W-FRI").last().dropna()
MM40 = R_IB.rolling(40).mean()
ESC = (R_IB > MM40).shift(1)                                   # decide com a semana anterior
RET_I = FECH["ITUB4.SA"].resample("W-FRI").last().pct_change(fill_method=None)
RET_B = FECH["BBDC4.SA"].resample("W-FRI").last().pct_change(fill_method=None)
OK = ESC.notna() & MM40.shift(1).notna() & RET_I.notna() & RET_B.notna()
TROCA = np.where(ESC[OK].astype(bool), RET_I[OK], RET_B[OK])
MEIO = 0.5 * RET_I[OK] + 0.5 * RET_B[OK]
ANOS3 = OK.sum() / 52
CRESC = lambda r: (1 + pd.Series(r)).prod() ** (1 / ANOS3) - 1
print(f"   Par ITUB4 / BBDC4 desde {OK[OK].index[0]:%Y}: ficar com a mais forte (razão acima da média de 40 semanas)"
      f" {ps(CRESC(TROCA))} ao ano | metade em cada {ps(CRESC(MEIO))} | só ITUB4 {ps(CRESC(RET_I[OK]))} | só BBDC4"
      f" {ps(CRESC(RET_B[OK]))}; trocas: {int((ESC[OK].astype(int).diff().abs() > 0).sum())}")

## Parte 4: Mansfield

In [ ]:
RS = SEMANAL[TICK].div(SEMANAL["^BVSP"], axis=0)             # linha de força relativa (ação / índice)
MANSFIELD = (RS / RS.rolling(52).mean() - 1) * 100            # acima de zero = mais forte que o normal
HOJE = MANSFIELD.iloc[-1].dropna().sort_values()
print(f"\nParte 4. Força relativa de Mansfield (52 semanas) em {FECH.index[-1]:%d/%m/%Y}, {len(HOJE)} ações")
print("   Mais fortes: " + ", ".join(f"{k[:-3]} {v:+.0f}" for k, v in HOJE.tail(5)[::-1].items()))
print("   Mais fracas: " + ", ".join(f"{k[:-3]} {v:+.0f}" for k, v in HOJE.head(5).items()))
print(f"   Acima de zero: {pc((HOJE > 0).mean(), 0)} das ações")

## Parte 5: momentum relativo 12-1

In [ ]:
ACM = MENSAL[TICK]
RET = ACM.pct_change(fill_method=None)
SINAL = ACM.shift(2) / ACM.shift(13) - 1                       # 12 meses até o mês anterior, pulando o mais recente
CART = []
for data in RET.index[13:]:
    s = SINAL.loc[data].dropna()
    r = RET.loc[data, s.index]
    if len(s) < 25:
        continue
    q = pd.qcut(s.rank(method="first"), 5, labels=False)      # quintis: 4 = líderes, 0 = lanternas
    CART.append({"data": data, "lideres": r[q == 4].mean(), "lanternas": r[q == 0].mean(), "todas": r.mean()})
CART = pd.DataFrame(CART).set_index("data").dropna()
ANOS = len(CART) / 12
TAB_MOM = []
print(f"\nParte 5. Momentum 12-1 na B3, carteiras mensais de mesmo peso, {CART.index[0]:%m/%Y} a {CART.index[-1]:%m/%Y}")
for col, nome in [("lideres", "20% líderes"), ("todas", "todas as ações"), ("lanternas", "20% lanternas")]:
    acum = (1 + CART[col]).cumprod()
    cresc = acum.iloc[-1] ** (1 / ANOS) - 1
    queda = (acum / acum.cummax() - 1).min()
    TAB_MOM.append((nome, cresc, queda))
    print(f"   {nome:<15} {ps(cresc)} ao ano | pior queda {pc(queda, 0)}")
GANHA = (CART["lideres"] > CART["lanternas"]).mean()
print(f"   Líderes à frente das lanternas em {pc(GANHA, 0)} dos meses")

In [ ]:
import matplotlib.pyplot as plt
(100 * (1 + CART).cumprod()).plot(figsize=(8, 3.5), logy=True, color=["black", "silver", "gray"])
plt.title("Momentum 12-1 na B3: líderes, lanternas e todas")
plt.show()

## Parte 6: grade de nove didática

In [ ]:
LIM = 0.03                                                     # +-3% em 13 semanas = "de lado"
P13 = SEMANAL[TICK].pct_change(13, fill_method=None)
RS13 = RS.pct_change(13, fill_method=None)
LINHA = np.select([P13 > LIM, P13 < -LIM], [2, 0], 1)          # 2 = preço subindo, 0 = caindo
COLUNA = np.select([RS13 > LIM, RS13 < -LIM], [2, 0], 1)       # 2 = batendo o índice, 0 = perdendo
CASA = pd.DataFrame(LINHA * 3 + COLUNA + 1, index=P13.index, columns=P13.columns)  # 9 = sobe e bate o índice
CASA = CASA.where(P13.notna() & RS13.notna())
FUT = RS.shift(-13) / RS - 1                                   # força relativa nas 13 semanas seguintes
TAB = pd.DataFrame({"casa": CASA.stack(), "depois": FUT.stack()}).dropna()
NOMES = {9: "preço sobe, bate o índice", 8: "preço sobe, empata", 7: "preço sobe, perde do índice",
         6: "preço de lado, bate o índice", 5: "preço de lado, empata", 4: "preço de lado, perde",
         3: "preço cai, bate o índice", 2: "preço cai, empata", 1: "preço cai, perde do índice"}
TAB_GRADE = []
print(f"\nParte 6. Grade de nove didática (13 semanas, faixa de +-3%), {nr(len(TAB), 0)} observações semanais")
for k in range(9, 0, -1):
    x = TAB.loc[TAB["casa"] == k, "depois"]
    TAB_GRADE.append((k, NOMES[k], len(x), x.mean(), (x > 0).mean()))
    print(f"   casa {k} ({NOMES[k]:<28}) {len(x):>6} obs. | força relativa 13 semanas depois: média {ps(x.mean())},"
          f" bateu o índice em {pc((x > 0).mean(), 0)}")
ULT = CASA.iloc[-1].dropna()
print(f"   Em {FECH.index[-1]:%d/%m/%Y}: {pc((ULT >= 7).mean(), 0)} das ações nas casas 7 a 9 e {pc((ULT <= 3).mean(), 0)}"
      f" nas casas 1 a 3")

## Parte 7: o fôlego da força relativa

In [ ]:
FOLEGO = (MANSFIELD > 0).sum(axis=1) / MANSFIELD.notna().sum(axis=1)
FOLEGO = FOLEGO[MANSFIELD.notna().sum(axis=1) >= 30]
IB_S = SEMANAL["^BVSP"]
R26 = IB_S.shift(-26) / IB_S - 1
F = pd.DataFrame({"folego": FOLEGO, "depois": R26}).dropna()
BAIXO, ALTO = F[F["folego"] < 0.3], F[F["folego"] > 0.7]
print(f"\nParte 7. Ações com Mansfield acima de zero: hoje {pc(FOLEGO.iloc[-1], 0)}; média desde {FOLEGO.index[0]:%Y}:"
      f" {pc(FOLEGO.mean(), 0)}")
print(f"   Ibovespa nas 26 semanas seguintes: com menos de 30% das ações fortes {ps(BAIXO['depois'].mean())}"
      f" ({len(BAIXO)} semanas) | com mais de 70% {ps(ALTO['depois'].mean())} ({len(ALTO)} semanas) | todas"
      f" {ps(F['depois'].mean())}")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a Parte 3: a função `razao`, a média de 40 semanas (`MM40`) e a linha `ESC = (R_IB > MM40).shift(1)`. Mostre por que o `shift(1)` faz a regra decidir com a semana anterior, e não com a semana que ela vai medir." Confira com a Parte 3, que imprime, para o par ITUB4 / BBDC4, 9,7% ao ano na troca, com 78 trocas, 11,0% com metade em cada ação e 13,1% só com ITUB4.
2. **Adaptar:** "Numa célula nova, depois da Parte 4, baixe [ATIVO] do Yahoo Finance com `auto_adjust=True`, como a célula de abertura, e calcule a linha de força relativa contra o `^BVSP` e o Mansfield de 52 semanas com as mesmas contas da Parte 4, em dados semanais de sexta-feira. Não mude a lista `ACOES` nem a fórmula." Antes de olhar o ativo novo, confira se a Parte 4 continua mostrando UGPA3 (+32) como a mais forte e 33% das ações acima de zero, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 3, troque a média de 40 semanas por 20 e por 60. A regra de trocar o pangaré pelo alazão passa a metade em cada?
2. Na Parte 5, troque 12 meses por 6. As líderes continuam à frente?
3. Na Parte 6, use 26 semanas no lugar de 13. Alguma casa da grade passa a prever melhor?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.